In [1]:
from prefect import flow, task, get_run_logger
import time

# ---------------------------------------------------------
# 1. สร้าง Tasks (งานย่อย)
# ---------------------------------------------------------

# ตั้งค่าให้ Task นี้ลองทำใหม่ได้ 2 ครั้ง ถ้าระหว่างทำเกิด Error (เช่น เน็ตหลุด)
@task(retries=2, retry_delay_seconds=3)
def extract_data():
    logger = get_run_logger()
    logger.info("กำลังดึงข้อมูลจาก API...")
    time.sleep(2) # จำลองว่าใช้เวลาดึงข้อมูล 2 วินาที
    return {"status": "success", "data": [10, 20, 30, 40, 50]}

@task
def transform_data(raw_data):
    logger = get_run_logger()
    logger.info("กำลังคำนวณและแปลงข้อมูล...")
    
    # เอาตัวเลขในลิสต์มาคูณ 2
    numbers = raw_data.get("data", [])
    transformed = [n * 2 for n in numbers]
    return transformed

@task
def load_data(final_data):
    logger = get_run_logger()
    logger.info(f"บันทึกข้อมูลสำเร็จ! ผลลัพธ์คือ: {final_data}")
    return True

# ---------------------------------------------------------
# 2. สร้าง Flow (ร้อยเรียง Pipeline)
# ---------------------------------------------------------

# log_prints=True จะทำให้คำสั่ง print() ธรรมดา เข้าไปอยู่ในระบบ Log ของ Prefect ด้วย
@flow(name="My First Data Pipeline", log_prints=True)
def my_test_pipeline():
    print("🚀 เริ่มรัน Pipeline ทดสอบระบบ!")
    
    # สั่งให้ Task ทำงานต่อกัน (ส่งผลลัพธ์จากตัวนึง ไปอีกตัวนึง)
    raw = extract_data()
    clean = transform_data(raw)
    result = load_data(clean)
    
    print("✅ Pipeline ทำงานเสร็จสมบูรณ์!")
    return result

# ---------------------------------------------------------
# 3. จุดสั่งรัน
# ---------------------------------------------------------
if __name__ == "__main__":
    my_test_pipeline()
    my_test_pipeline.serve(name="Test-Deployment", work_pool_name="default-pool")

09:18:45.246 | INFO    | Flow run 'tremendous-toad' - Beginning flow run 'tremendous-toad' for flow 'My First Data Pipeline'

09:18:45.264 | INFO    | Flow run 'tremendous-toad' - View at http://prefect-server:4200/runs/flow-run/6ec329ce-1649-47e5-a80a-b6e2ac5523ec

09:18:45.270 | INFO    | Flow run 'tremendous-toad' - 🚀 เริ่มรัน Pipeline ทดสอบระบบ!

09:18:45.352 | INFO    | Task run 'extract_data-6b4' - กำลังดึงข้อมูลจาก API...

09:18:47.360 | INFO    | Task run 'extract_data-6b4' - Finished in state Completed()

...

09:18:47.386 | INFO    | Task run 'load_data-b7d' - บันทึกข้อมูลสำเร็จ! ผลลัพธ์คือ: [20, 40, 60, 80, 100]

09:18:47.392 | INFO    | Task run 'load_data-b7d' - Finished in state Completed()

09:18:47.400 | INFO    | Flow run 'tremendous-toad' - ✅ Pipeline ทำงานเสร็จสมบูรณ์!

09:18:48.322 | INFO    | Flow run 'tremendous-toad' - Finished in state Completed()

TypeError: Flow.serve() got an unexpected keyword argument 'work_pool_name'